
# RAM, CPU, and OS: Hardware Mechanisms vs. Software Abstractions

## What Is Actually Stored in RAM?

RAM fundamentally stores bytes.

It does not know whether those bytes represent instructions, variables, stacks, or page tables.

A typical operating system uses RAM for the following purposes:

| Contents | Purpose | Who Defines the Meaning? |
|---|---|---|
| Kernel Code | OS machine instructions | Compiler, Linker, OS |
| User Program Code | Application machine instructions | Compiler, Linker, OS |
| Stack | Function calls, local variables, saved state | Compiler, ABI, Runtime |
| Heap | Dynamically allocated objects | Allocator, Runtime, OS |
| Data (.data) | Initialized global/static variables | Compiler, Linker, Loader |
| BSS (.bss) | Zero-initialized global/static variables | Compiler, Linker, Loader |
| Read-only Data (.rodata) | Constants, string literals | Compiler, Linker, Loader |
| Page Tables | Virtual-to-physical address mappings | OS defines mappings; CPU defines format |
| OS Metadata | Process tables, file descriptors, scheduling state | OS |
| Buffers / Software Caches | Temporary copies of disk/network data | OS, Applications |
| Free Pages | Memory available for future allocation | OS Memory Manager |
| Reserved Memory | Memory reserved for special purposes | Platform, Firmware, OS |

These are **logical categories, not hardware-defined RAM partitions**.

Physical pages belonging to different categories may be scattered throughout RAM.

Important distinctions:

- Both kernel and user programs need code, data, and stacks.
- Each thread typically has its own stack.
- A function call creates a stack frame, not an entirely new stack.
- The kernel also dynamically allocates memory.
- A heap is a software-managed region, not a hardware feature.
- Free memory is still physical RAM; it simply has no current allocation.
- Some RAM may also be used for DMA, shared memory, or firmware-related purposes.

### Special Case: Page Tables

Page tables are different from most other entries.

They are stored in RAM, but their format is defined by the CPU architecture.

The OS creates and updates them.

When virtual memory is enabled, the CPU's MMU interprets these page tables to translate addresses and check permissions.

Therefore:

**Page Tables = Software-managed data structures with hardware-defined meaning.**

---

## Hardware Mechanisms vs. Software Abstractions

### A. What the CPU Actually Understands

| Hardware Concept | CPU Understands It? | Explanation |
|---|---|---|
| Machine Instructions | Yes | Executes ADD, LOAD, STORE, JUMP, etc. |
| Registers | Yes | Hardware storage inside CPU |
| Program Counter (PC) | Yes | Address of the instruction being executed |
| Physical Addresses | Yes | Used to access physical memory/devices |
| Virtual Addresses | Yes, with MMU support | Translated into physical addresses |
| Page Table Format | Yes, with paging support | MMU interprets page-table entries |
| Privilege Modes | Yes | Machine/Supervisor/User modes in RISC-V |
| Interrupts / Exceptions | Yes | Hardware-defined control transfers |
| Timer Interrupts | Yes, with timer hardware | Allow software to regain CPU control |

These mechanisms are defined by the hardware architecture.

Software must obey their rules.

### B. What Software Creates

| Software Concept | CPU Understands It? | Implemented By |
|---|---|---|
| Function | No | Compiler + Calling Convention |
| Stack Frame | No | Compiler + ABI |
| Heap | No | Memory Allocator |
| `.text` / `.data` / `.bss` | No | Compiler + Linker + Loader |
| Process | No | OS |
| Thread | No | OS / Runtime |
| Scheduler | No | OS |
| File / Directory | No | File System |
| File Descriptor | No | OS |
| System Call API | Not as an API | OS, using hardware trap mechanisms |
| Kernel / User Program Identity | No | OS; enforced using hardware permissions |
| Buffer Cache | No | OS |
| ELF Format | No | Toolchain + Loader |

A CPU does not know what a process is.

It only knows registers, execution state, memory addresses, instructions, permissions, and interrupts.

The OS combines these hardware mechanisms to create the illusion of independent processes.

---

## How Software Builds OS Abstractions

| Hardware Mechanism | Software Implementation | Resulting Abstraction |
|---|---|---|
| Registers + Jumps | Calling Convention | Functions |
| Registers + RAM | Stack Management | Call Stack |
| RAM + Memory Instructions | Allocator | Heap |
| MMU + Page Tables | Virtual Memory Manager | Per-process Address Space |
| Privilege Modes + Traps | Kernel/System Call Handling | Protected OS Services |
| Timer Interrupts | Scheduler + Context Switching | Preemptive Multitasking |
| Storage Device I/O | Drivers + File System | Files and Directories |
| CPU State + Memory + Scheduler | Process Management | Processes and Threads |

A few important observations:

1. **Virtual Memory:** The MMU provides address translation. The OS decides which pages belong to which processes.

2. **Process Isolation:** Hardware enforces access permissions, but the OS establishes the isolation policy.

3. **Scheduling:** A timer interrupt can give the OS an opportunity to run. The OS decides which thread executes next.

4. **File Systems:** Storage hardware provides block or byte access. Software defines how those bytes represent files.

5. **System Calls:** Hardware provides controlled transitions into privileged execution. The OS implements the services exposed through those transitions.